# Self-Reflection & Critique: Engineering Autonomous LLM Reflection
### Practical implementations of every concept from the BIA deck

**How LLMs diagnose, refine, and improve outputs before agents scale up.**


| # | Section | Slide concept |
|---|---------|---------------|
| 1 | Setup & helper functions | — |
| 2 | Self-Reflection vs Critique | *The useful unit is "what should change, and why?"* |


### 1. Setup & helper functions 

In [1]:
import os, json, re, textwrap
from dataclasses import dataclass, field
from typing import Optional
from litellm import completion
from dotenv import load_dotenv



In [2]:
load_dotenv()

True

In [3]:
MODEL = "gpt-4o-mini"  

In [4]:
def llm(prompt: str, system: str = "You are a helpful assistant.",
        temperature: float = 0.7, json_mode: bool = False) -> str:
    """Single LLM call. json_mode=True nudges + parses strict JSON output."""
    messages = [{"role": "system", "content": system},
                {"role": "user", "content": prompt}]
    resp = completion(model=MODEL, messages=messages, temperature=temperature)
    return resp.choices[0].message.content.strip()


In [5]:
def llm_json(prompt: str, system: str, temperature: float = 0.0) -> dict:
    """LLM call that must return JSON. Strips markdown fences and parses."""
    raw = llm(prompt, system=system + "\nRespond ONLY with valid JSON. No preamble, no markdown fences.",
              temperature=temperature)
    cleaned = re.sub(r"^```(?:json)?|```$", "", raw.strip(), flags=re.MULTILINE).strip()
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        # one repair attempt — a common production trick
        repaired = llm(f"Fix this into strict valid JSON, output only the JSON:\n{raw}",
                       system="You repair malformed JSON.", temperature=0.0)
        repaired = re.sub(r"^```(?:json)?|```$", "", repaired.strip(), flags=re.MULTILINE).strip()
        return json.loads(repaired)


def show(text, width=100):
    print(textwrap.fill(text, width=width, replace_whitespace=False))

In [6]:
show(llm("Hi", "You are helpful assitance to answer any query"))

Hello! How can I assist you today?


In [7]:
BRIEF = "Write a short email inviting working professionals to a weekend GenAI workshop"

first_draft = llm(f"Write an email for this: {BRIEF}", temperature=1.0)
show(first_draft)


Subject: Join Us for a Weekend GenAI Workshop!

Dear [Recipient's Name],

I hope this message finds
you well.

We are excited to invite you to our upcoming GenAI Workshop, taking place over the
weekend on [insert dates] at [insert location]. This workshop is designed specifically for working
professionals eager to explore the latest advancements in Generative AI and how they can be applied
in various industries.

Whether you’re looking to enhance your skill set, gain practical insights,
or network with like-minded individuals, this workshop offers a valuable opportunity to dive into
the world of GenAI.

**Details:**
- **Date:** [Insert dates]
- **Time:** [Insert time]
-
**Location:** [Insert location]
- **Registration Fee:** [Insert fee, if applicable]

Please RSVP by
[insert RSVP deadline] to secure your spot, as space is limited. You can respond to this email or
register directly at [insert registration link].

We hope to see you there!

Best regards,

[Your
Name]  
[Your Position]  

---
## 3. Self-Reflection vs Critique — precise definitions

The deck draws a jigsaw between two interlocking concepts:

- **Self-reflection** — *an LLM-generated review of an output against the task goal, constraints, and quality criteria.* (The act of looking.)
- **Critique** — *a diagnosis that names specific defects and gives revision instructions.* (The actionable artifact.)

> **The useful unit is not "confidence." The useful unit is "what should change, and why?"**

A reflection that says *"This looks good, 8/10, I'm fairly confident"* is useless to a refiner. A critique that says *"The subject line doesn't state the date; move '25 July' into it"* is directly executable. Let's demonstrate the difference on the same draft.


In [8]:
# A confidence-style reflection (what we DON'T want)
confidence_reflection = llm(
    f"Review this email and say how confident you are that it is good:\n\n{first_draft}",
    temperature=0.3)


print("=== CONFIDENCE-STYLE REFLECTION (weak) ===")
show(confidence_reflection)

=== CONFIDENCE-STYLE REFLECTION (weak) ===
I would rate my confidence in this email as high, around 85-90%. It effectively communicates the
essential details of the workshop and maintains a professional tone. Here are some strengths and
areas for improvement:

### Strengths:
1. **Clear Subject Line**: The subject line is engaging and
clearly states the purpose of the email.
2. **Professional Tone**: The email maintains a friendly
yet professional tone throughout.
3. **Structured Information**: The details are well-organized,
making it easy for the recipient to find important information quickly.
4. **Call to Action**: The
email includes a clear call to action regarding RSVP and registration.

### Areas for Improvement:
1. **Personalization**: Ensure that the recipient's name is inserted correctly to make it more
personal.
2. **Specific Details**: Make sure to fill in all placeholders (dates, location, time,
registration fee, RSVP deadline, registration link) before sending.
3. **Additi

In [9]:
change_critique = llm(
    f"""Here is a brief and a draft.

BRIEF:
{BRIEF}

DRAFT:
{first_draft}

List the specific defects (what is missing, incorrect, vague, or risky relative to the brief),
and for each defect give a concrete revision instruction a writer could apply mechanically.""",
    temperature=0.3)

print("=== CHANGE-ORIENTED CRITIQUE (strong) ===")
show(change_critique)

=== CHANGE-ORIENTED CRITIQUE (strong) ===
Here are the specific defects identified in the draft along with concrete revision instructions:

1.
**Missing Specific Dates and Location**:
   - **Defect**: The placeholders for dates and location
are not filled in.
   - **Revision Instruction**: Replace "[insert dates]" and "[insert location]"
with the actual dates and location of the workshop.

2. **Vague Time Specification**:
   -
**Defect**: The time of the workshop is not specified.
   - **Revision Instruction**: Replace
"[insert time]" with the actual start and end times of the workshop.

3. **Unclear Registration
Fee**:
   - **Defect**: The registration fee is left as a placeholder.
   - **Revision
Instruction**: Replace "[insert fee, if applicable]" with the actual fee amount or state "Free" if
there is no fee.

4. **Ambiguous RSVP Deadline**:
   - **Defect**: The RSVP deadline is not
provided.
   - **Revision Instruction**: Replace "[insert RSVP deadline]" with the actual date by
whi